# Notebook 2: MobileNetV2 Training

**Image Manipulation Detection System — Google Colab (Main Training Notebook)**

This notebook trains the MobileNetV2 binary classifier to distinguish
**Authentic** from **Potentially Manipulated** images using CASIA v1.0 and v2.0.

---

**Prerequisites:**
- Run `01_dataset_preparation.ipynb` first to generate `data/metadata/dataset_split.csv`.
- Enable **GPU runtime**: Runtime → Change runtime type → GPU

**Outputs produced:**
- `artifacts/models/mobilenetv2.keras` — trained model file
- `artifacts/models/model_config.json` — architecture configuration
- `artifacts/models/class_labels.json` — class mapping
- `artifacts/models/preprocessing_config.json` — preprocessing parameters
- `artifacts/plots/mobilenet_training_history.csv` — per-epoch metrics
- `artifacts/plots/training_curves.png` — accuracy and loss plots

**Two-phase training strategy:**
1. **Phase 1 — Feature Extraction** (frozen backbone): only the classification head is trained.
2. **Phase 2 — Fine-Tuning** (partial unfreeze): the top 30 MobileNetV2 layers are unfrozen
   and trained with a reduced learning rate.

> **Important:** Do NOT use this notebook to run inference. The Flask application
> handles inference locally using the downloaded `.keras` file.

## Step 1: Clone Repository and Install Dependencies

In [ ]:
# TODO: Replace with your actual GitHub repository URL and repo folder name
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
REPO_NAME = "YOUR_REPO"

import os

if not os.path.exists(REPO_NAME):
    print(f"Cloning {GITHUB_REPO_URL} ...")
    os.system(f"git clone {GITHUB_REPO_URL}")
else:
    print(f"Repository folder '{REPO_NAME}' exists — pulling latest changes.")
    os.system(f"cd {REPO_NAME} && git pull")

os.chdir(REPO_NAME)
print("Working directory:", os.getcwd())

In [ ]:
import subprocess
result = subprocess.run(
    ["pip", "install", "-r", "requirements.txt", "-q"],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else "Installation complete.")
if result.returncode != 0:
    print("ERRORS:", result.stderr[-500:])

## Step 2: GPU Availability Check

Training on CPU is possible but very slow (~10x slower than GPU).
If no GPU is detected, go to **Runtime → Change runtime type → GPU**.

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

gpu_available = len(tf.config.list_physical_devices("GPU")) > 0
if gpu_available:
    print("\nGPU detected. Training will use GPU acceleration.")
    # Allow GPU memory growth to avoid OOM errors
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("\nWARNING: No GPU detected. Training will be significantly slower on CPU.")
    print("Go to Runtime -> Change runtime type -> GPU to enable GPU.")

## Step 3: Imports and Project Configuration

In [ ]:
import sys
import json
import os
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for Colab

# Add project root to Python path
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import Config
from src.models.mobilenet_model import build_mobilenet_model, unfreeze_mobilenet_layers
from src.training.train_mobilenet import build_dataset_frame, build_training_datasets
from src.preprocessing.image_normalizer import normalize_image

print("Config loaded.")
print("  Input shape             :", Config.INPUT_SHAPE)
print("  Batch size              :", Config.BATCH_SIZE)
print("  Normalization mode      :", Config.MOBILENET_NORMALIZATION_MODE)
print("  Warmup epochs           :", Config.MOBILENET_WARMUP_EPOCHS)
print("  Fine-tune epochs        :", Config.MOBILENET_FINE_TUNE_EPOCHS)
print("  Layers to unfreeze      :", Config.MOBILENET_LAYERS_TO_UNFREEZE)
print("  Initial learning rate   :", Config.MOBILENET_INITIAL_LEARNING_RATE)
print("  Fine-tune learning rate :", Config.MOBILENET_FINE_TUNE_LEARNING_RATE)

## Step 4: Configure Paths

If you supplied the CASIA dataset to a custom location, override the path variables below.

In [ ]:
# Dataset split CSV produced by Notebook 1
SPLIT_CSV_PATH = Config.DATASET_SPLIT_PATH  # data/metadata/dataset_split.csv

# Model output paths
ARTIFACTS_DIR = Config.ARTIFACTS_DIR
MODEL_DIR = Config.MODEL_DIR
PLOTS_DIR = Config.PLOTS_DIR
MODEL_SAVE_PATH = MODEL_DIR / "mobilenetv2.keras"

# Checkpoint directory — Colab's /content is preserved for the session duration
CHECKPOINT_DIR = Path("/content/checkpoints")
CHECKPOINT_PATH = CHECKPOINT_DIR / "mobilenetv2_best.keras"

# Create all required directories
for d in [MODEL_DIR, PLOTS_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Split CSV     :", SPLIT_CSV_PATH)
print("Split exists  :", SPLIT_CSV_PATH.exists())
print("Model will be saved to:", MODEL_SAVE_PATH)
print("Checkpoint dir:", CHECKPOINT_DIR)

if not SPLIT_CSV_PATH.exists():
    raise FileNotFoundError(
        f"Dataset split CSV not found at {SPLIT_CSV_PATH}.\n"
        "Please run 01_dataset_preparation.ipynb first."
    )

## Step 5: Load Dataset

Images are loaded from the paths recorded in `dataset_split.csv` and normalized
using the MobileNetV2 preprocessing mode (pixel values scaled to [−1, 1]).

> **Normalization formula:** `pixel / 127.5 − 1.0`
>
> This matches `tensorflow.keras.applications.mobilenet_v2.preprocess_input`
> and must be applied identically during Flask inference.

In [ ]:
print("Loading training and validation datasets ...")
print("(This may take several minutes depending on dataset size.)")

try:
    train_dataset, validation_dataset = build_training_datasets(SPLIT_CSV_PATH)
    print("Datasets loaded successfully.")

    # Count batches for reference
    train_batches = sum(1 for _ in train_dataset)
    val_batches   = sum(1 for _ in validation_dataset)
    print(f"  Training batches  : {train_batches} (batch size {Config.BATCH_SIZE})")
    print(f"  Validation batches: {val_batches}")
except FileNotFoundError as e:
    raise RuntimeError(
        f"Dataset loading failed: {e}\n"
        "Ensure dataset_split.csv exists and all image paths in it are reachable from Colab."
    ) from e

## Step 6: Data Augmentation

Light augmentation is applied only to the training set to improve generalization.
Augmentation is kept conservative to avoid distorting ELA-relevant compression artifacts.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),   # ±5% rotation — conservative
    tf.keras.layers.RandomZoom(0.05),        # ±5% zoom
], name="data_augmentation")

def augment_batch(images, labels):
    """Apply augmentation to a training batch."""
    return data_augmentation(images, training=True), labels

# Apply augmentation only to training data
train_dataset = train_dataset.map(
    augment_batch,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

print("Augmentation pipeline configured:")
print("  - RandomFlip (horizontal)")
print("  - RandomRotation (±5%)")
print("  - RandomZoom (±5%)")
print("Augmentation is applied to training batches only.")

## Step 7: Build MobileNetV2 Model

Architecture:
- **Backbone:** MobileNetV2 pretrained on ImageNet (frozen during Phase 1)
- **Head:** GlobalAveragePooling2D → Dense(256, relu) → Dropout(0.4) → Dense(128, relu) → Dropout(0.3) → Dense(1, sigmoid)
- **Loss:** Binary cross-entropy
- **Output:** Sigmoid probability; ≥ 0.5 → Manipulated

In [ ]:
print("Building MobileNetV2 model ...")
model = build_mobilenet_model(input_shape=Config.INPUT_SHAPE)

# Confirm backbone is frozen
base = model.get_layer(Config.MOBILENET_BASE_MODEL_NAME)
trainable_params   = sum(p.numpy().size for p in model.trainable_weights)
total_params       = sum(p.numpy().size for p in model.weights)
print(f"  Trainable parameters : {trainable_params:,}")
print(f"  Total parameters     : {total_params:,}")
print(f"  Backbone trainable   : {base.trainable}  (expected: False)")
model.summary()

## Step 8: Phase 1 — Feature Extraction (Frozen Backbone)

Only the classification head is trained. The MobileNetV2 backbone weights
remain frozen, using high-level ImageNet features as a fixed feature extractor.

**Checkpoint recovery:** If this Colab session disconnects, the best weights
are preserved at `/content/checkpoints/mobilenetv2_best.keras`.
To resume, reload the checkpoint in the recovery cell below.

In [ ]:
# --- Phase 1 Callbacks ---
phase1_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(CHECKPOINT_PATH),
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)
phase1_csv_logger = tf.keras.callbacks.CSVLogger(
    filename=str(PLOTS_DIR / "phase1_training_history.csv")
)
phase1_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=Config.MOBILENET_EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    phase1_checkpoint,
    phase1_csv_logger,
]
print("Phase 1 callbacks configured.")
print(f"  Checkpoint saved to: {CHECKPOINT_PATH}")

In [ ]:
print(f"Starting Phase 1 — Feature Extraction ({Config.MOBILENET_WARMUP_EPOCHS} epochs max) ...")
phase1_history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=Config.MOBILENET_WARMUP_EPOCHS,
    callbacks=phase1_callbacks,
    verbose=1,
)
print("Phase 1 complete.")
print(f"Best checkpoint saved to: {CHECKPOINT_PATH}")

## Checkpoint Recovery

If your Colab session disconnected between Phase 1 and Phase 2, run this cell
to reload the best Phase 1 checkpoint before continuing.

**Skip this cell if you are continuing from an uninterrupted session.**

In [ ]:
# SKIP THIS CELL if Phase 1 completed in this session without disconnection.
# Run this cell ONLY if you need to reload from a saved checkpoint.

CHECKPOINT_TO_LOAD = CHECKPOINT_PATH  # or set a full path to a specific .keras file

if CHECKPOINT_TO_LOAD.exists():
    print(f"Loading checkpoint from {CHECKPOINT_TO_LOAD} ...")
    model = tf.keras.models.load_model(str(CHECKPOINT_TO_LOAD))
    print("Checkpoint loaded successfully.")
    # Reset phase1_history if resuming mid-session
    class _PlaceholderHistory:
        history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}
    phase1_history = _PlaceholderHistory()
else:
    print(f"No checkpoint found at {CHECKPOINT_TO_LOAD}.")
    print("Run Phase 1 training (Step 8) first.")

## Step 9: Phase 2 — Fine-Tuning (Partial Unfreeze)

The top 30 MobileNetV2 layers are unfrozen and trained with a reduced learning
rate (1e-5). The classification head continues to train.

**Validation loss is used for model selection** — the best checkpoint across both
phases is saved as the final `mobilenetv2.keras` model.

In [ ]:
print(f"Unfreezing top {Config.MOBILENET_LAYERS_TO_UNFREEZE} MobileNetV2 backbone layers ...")
model = unfreeze_mobilenet_layers(
    model,
    num_layers_to_unfreeze=Config.MOBILENET_LAYERS_TO_UNFREEZE,
    fine_tune_lr=Config.MOBILENET_FINE_TUNE_LEARNING_RATE,
)

base = model.get_layer(Config.MOBILENET_BASE_MODEL_NAME)
trainable_params = sum(p.numpy().size for p in model.trainable_weights)
total_params     = sum(p.numpy().size for p in model.weights)
print(f"  Trainable parameters : {trainable_params:,}")
print(f"  Total parameters     : {total_params:,}")
print(f"  Fine-tune LR         : {Config.MOBILENET_FINE_TUNE_LEARNING_RATE}")

In [ ]:
# Phase 2 checkpoint path — overwrites phase 1 checkpoint if val_loss improves
phase2_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(CHECKPOINT_PATH),
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)
phase2_csv_logger = tf.keras.callbacks.CSVLogger(
    filename=str(PLOTS_DIR / "phase2_training_history.csv")
)
phase2_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=Config.MOBILENET_EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    phase2_checkpoint,
    phase2_csv_logger,
]

print(f"Starting Phase 2 — Fine-Tuning ({Config.MOBILENET_FINE_TUNE_EPOCHS} epochs max) ...")
phase2_history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=Config.MOBILENET_FINE_TUNE_EPOCHS,
    callbacks=phase2_callbacks,
    verbose=1,
)
print("Phase 2 complete.")

## Step 10: Training History Plots

In [ ]:
def _merge_histories(h1, h2, metric):
    """Concatenate two Keras history lists for a given metric."""
    vals1 = h1.history.get(metric, [])
    vals2 = h2.history.get(metric, [])
    return vals1 + vals2

acc     = _merge_histories(phase1_history, phase2_history, "accuracy")
val_acc = _merge_histories(phase1_history, phase2_history, "val_accuracy")
loss    = _merge_histories(phase1_history, phase2_history, "loss")
val_loss= _merge_histories(phase1_history, phase2_history, "val_loss")
phase1_len = len(phase1_history.history.get("loss", []))

if acc and loss:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(acc) + 1)

    # Accuracy subplot
    axes[0].plot(epochs, acc, label="Train Accuracy")
    axes[0].plot(epochs, val_acc, label="Val Accuracy")
    if phase1_len:
        axes[0].axvline(x=phase1_len, color="gray", linestyle="--", alpha=0.6, label="Fine-tune start")
    axes[0].set_title("Model Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Loss subplot
    axes[1].plot(epochs, loss, label="Train Loss")
    axes[1].plot(epochs, val_loss, label="Val Loss")
    if phase1_len:
        axes[1].axvline(x=phase1_len, color="gray", linestyle="--", alpha=0.6, label="Fine-tune start")
    axes[1].set_title("Model Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Binary Cross-Entropy")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle("MobileNetV2 Training History (Phase 1 + Phase 2)", fontsize=13)
    plt.tight_layout()
    plot_path = PLOTS_DIR / "training_curves.png"
    plt.savefig(plot_path, dpi=Config.PLOT_DPI, bbox_inches="tight")
    plt.show()
    print(f"Plot saved to: {plot_path}")
else:
    print("No training history to plot (session may have been resumed from checkpoint).")

## Step 11: Validation Set Evaluation

The validation set is used here only to confirm final model quality.
It was **not** used to train weights — only to select the best checkpoint.
Definitive test-set evaluation is performed in Notebook 3.

In [ ]:
print("Evaluating on validation set ...")
val_results = model.evaluate(validation_dataset, verbose=1)
metric_names = model.metrics_names

print("\nValidation results:")
for name, value in zip(metric_names, val_results):
    print(f"  {name}: {value:.4f}")

## Step 12: Save the Trained Model

The model is saved as `mobilenetv2.keras` (TF native Keras format).
This file is what you download and place in `artifacts/models/` for local Flask inference.

In [ ]:
print(f"Saving model to: {MODEL_SAVE_PATH}")
model.save(str(MODEL_SAVE_PATH))
model_size_bytes = MODEL_SAVE_PATH.stat().st_size
model_size_mb = model_size_bytes / (1024 ** 2)
print(f"Model saved successfully. Size: {model_size_mb:.1f} MB")

## Step 13: Save Configuration Files

These JSON files record the exact preprocessing and architecture settings used
during training. They must accompany the `.keras` file to ensure consistent
inference in Flask.

In [ ]:
timestamp = datetime.now(timezone.utc).isoformat()

# model_config.json
model_config = {
    "model_name": Config.MOBILENET_MODEL_NAME,
    "architecture": "MobileNetV2",
    "input_shape": list(Config.INPUT_SHAPE),
    "normalization_mode": Config.MOBILENET_NORMALIZATION_MODE,
    "normalization_range": "[-1, 1]",
    "num_classes": 2,
    "output_activation": "sigmoid",
    "classification_threshold": Config.CLASSIFICATION_THRESHOLD,
    "pretrained_weights": Config.MOBILENET_WEIGHTS,
    "fine_tune_layers": Config.MOBILENET_LAYERS_TO_UNFREEZE,
    "head_units": list(Config.MOBILENET_HEAD_UNITS),
    "head_dropout_rates": list(Config.MOBILENET_HEAD_DROPOUT_RATES),
    "initial_learning_rate": Config.MOBILENET_INITIAL_LEARNING_RATE,
    "fine_tune_learning_rate": Config.MOBILENET_FINE_TUNE_LEARNING_RATE,
    "warmup_epochs_configured": Config.MOBILENET_WARMUP_EPOCHS,
    "finetune_epochs_configured": Config.MOBILENET_FINE_TUNE_EPOCHS,
    "training_dataset": "CASIA v1.0 + v2.0",
    "model_file": "mobilenetv2.keras",
    "model_size_mb": round(model_size_mb, 2),
    "version": "1.0.0",
    "trained_at": timestamp,
    "tensorflow_version": tf.__version__,
}
config_path = MODEL_DIR / "model_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=2)
print(f"Saved: {config_path}")

# class_labels.json
class_labels = {
    "0": Config.CLASS_LABELS[0],
    "1": Config.CLASS_LABELS[1],
    "threshold": Config.CLASSIFICATION_THRESHOLD,
    "note": "Sigmoid output >= threshold is classified as label 1 (Manipulated)",
}
labels_path = MODEL_DIR / "class_labels.json"
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(class_labels, f, indent=2)
print(f"Saved: {labels_path}")

# preprocessing_config.json
preprocessing_config = {
    "target_size": list(Config.TARGET_SIZE),
    "normalization_mode": Config.MOBILENET_NORMALIZATION_MODE,
    "normalization_formula": "pixel / 127.5 - 1.0",
    "color_mode": "RGB",
    "input_dtype": "float32",
    "note": "Must match src/preprocessing/image_normalizer.py normalize_image(mode='mobilenet')",
}
preproc_path = MODEL_DIR / "preprocessing_config.json"
with open(preproc_path, "w", encoding="utf-8") as f:
    json.dump(preprocessing_config, f, indent=2)
print(f"Saved: {preproc_path}")

print("\nAll configuration files saved successfully.")

## Step 14: Download the Model from Colab

Run the cell below to download `mobilenetv2.keras` to your local machine.
Then place it in `<project-root>/artifacts/models/`.

See `scripts/download_model_instructions.md` for full deployment steps.

In [ ]:
# Download the trained model to your local machine
try:
    from google.colab import files
    print(f"Initiating download of: {MODEL_SAVE_PATH.name}")
    files.download(str(MODEL_SAVE_PATH))
    print("Download started. Check your browser's download folder.")
except ImportError:
    print("Not running in Colab — use the Files panel or scp to retrieve the file.")
    print(f"Model path: {MODEL_SAVE_PATH.resolve()}")

print()
print("After downloading, place the file at:")
print("  <project-root>/artifacts/models/mobilenetv2.keras")
print()
print("Then verify with:")
print("  python -c \"from src.models.model_loader import ModelLoader; "
      "ModelLoader.load_all_models(); print('Loaded:', not ModelLoader.is_fallback('mobilenet'))\"")

## Summary

| File | Description |
|------|-------------|
| `artifacts/models/mobilenetv2.keras` | Trained model — download this |
| `artifacts/models/model_config.json` | Architecture and training metadata |
| `artifacts/models/class_labels.json` | Class mapping (0=Authentic, 1=Manipulated) |
| `artifacts/models/preprocessing_config.json` | Normalization settings |
| `artifacts/plots/phase1_training_history.csv` | Per-epoch Phase 1 metrics |
| `artifacts/plots/phase2_training_history.csv` | Per-epoch Phase 2 metrics |
| `artifacts/plots/training_curves.png` | Training accuracy and loss curves |

**Next step:** Run `03_mobilenetv2_evaluation.ipynb` to evaluate the model
on the untouched test split.